In [ ]:
# General notebook settings
import logging
import warnings

import pypsa

warnings.filterwarnings("error", category=DeprecationWarning)
logging.getLogger("gurobipy").propagate = False
pypsa.options.params.optimize.log_to_console = False

# Effects: Multi-Quantity Accounting

This tutorial introduces **effects** — named tracked quantities such as CO$_2$ emissions, primary energy, land or water use. Components contribute to effects via coefficients — per carrier (columns on `n.carriers` such as `co2_emissions`) or per individual component (`marginal_effect_*` / `capital_effect_*` columns) — and one uniform mechanism covers everything you can do with them:

1. **report** any effect per component, per carrier, or over time via `n.statistics.effect`,
2. **bound** any effect with a global constraint of type `effect_limit` (the CO$_2$ budget as a special case),
3. **price** an effect into the objective (a carbon price as a declared input), and
4. **minimize** an effect instead of cost via `objective_effect`.

The guiding rule: an effect only enters the optimisation model if it is the objective, bounded, or priced. All other declared effects add nothing to the model — they are recomputed from the solution, so tracking extra quantities is free.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import pypsa

## A small demo system

Three generators over four days: cheap-but-dirty coal, flexible gas, and variable wind.

In [ ]:
def create_network():
    n = pypsa.Network(snapshots=pd.date_range("2026-01-01", periods=96, freq="h"))
    n.add("Carrier", "coal", co2_emissions=0.34, color="#9C5A1E", primary_energy=1.0)
    n.add("Carrier", "gas", co2_emissions=0.20, color="#E0A21C", primary_energy=1.0)
    n.add("Carrier", "wind", color="#2E7EB3")

    n.add("Bus", "electricity")
    n.add(
        "Generator",
        "coal plant",
        bus="electricity",
        carrier="coal",
        p_nom=90,
        efficiency=0.40,
        marginal_cost=30,
    )
    n.add(
        "Generator",
        "gas turbine",
        bus="electricity",
        carrier="gas",
        p_nom=100,
        efficiency=0.55,
        marginal_cost=55,
    )
    t = np.arange(96)
    n.add(
        "Generator",
        "wind farm",
        bus="electricity",
        carrier="wind",
        p_nom=120,
        marginal_cost=0.1,
        p_max_pu=np.clip(0.4 + 0.4 * np.sin(t / 7) + 0.15 * np.sin(t / 2.3), 0, 1),
    )
    n.add(
        "Load",
        "demand",
        bus="electricity",
        p_set=110 + 25 * np.cos(2 * np.pi * t / 24),
    )
    return n


n = create_network()

## Declaring effects

Effects are ordinary components. `carrier_attribute` names the column of `n.carriers` that holds the per-unit coefficients — `co2_emissions` is the column the CO$_2$ global constraint has always used, and any other column (here `primary_energy` in MWh$_{th}$ per MWh$_{th}$ of fuel) works the same way. Declaring them changes nothing about the optimisation.

In [ ]:
n.add("Effect", "co2", unit="t", carrier_attribute="co2_emissions")
n.add("Effect", "primary_energy", unit="MWh_th", carrier_attribute="primary_energy")
n.effects

In [ ]:
n.optimize(include_objective_constant=False);

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
dispatch = n.generators_t.p.rename(columns=n.generators.carrier)
colors = n.carriers.color[dispatch.columns]
dispatch.plot.area(ax=ax, color=colors.values, linewidth=0, alpha=0.9)
ax.set_ylabel("Dispatch [MW]")
ax.set_xlabel("")
ax.legend(title="", loc="upper left", ncols=3)
ax.margins(x=0)
plt.tight_layout()

## Reporting effects — system-wide, per asset, over time

`n.statistics.effect` reports any declared effect with the full statistics API: grouped by carrier, per individual asset (`groupby=False`), or time-resolved (`groupby_time=False`). Note that neither effect appears anywhere in the model — there is no CO$_2$ constraint here — yet both are fully accounted.

In [ ]:
n.statistics.effect("co2")

In [ ]:
n.statistics.effect("primary_energy", groupby=False)

In [ ]:
emissions_ts = n.statistics.effect("co2", groupby=False, groupby_time=False).T

fig, ax = plt.subplots(figsize=(9, 3))
gen_carriers = n.generators.carrier
for name in emissions_ts.columns.get_level_values("name"):
    series = emissions_ts.xs(name, axis=1, level="name").iloc[:, 0].fillna(0)
    ax.plot(series.index, series, color=n.carriers.color[gen_carriers[name]], label=name)
ax.set_ylabel("Emissions [t/h]")
ax.legend()
ax.margins(x=0)
plt.tight_layout()

## Bounding an effect

A global constraint of type `effect_limit` caps any effect; `carrier_attribute` names the effect. This generalizes the classic `primary_energy` CO$_2$ budget — for the built-in `co2` effect the two are exactly equivalent, including the shadow price `mu` (the implied carbon price).

In [ ]:
co2_unbounded = n.statistics.effect("co2").sum()

n_capped = create_network()
n_capped.add("Effect", "co2", unit="t", carrier_attribute="co2_emissions")
n_capped.add(
    "GlobalConstraint",
    "co2_budget",
    type="effect_limit",
    carrier_attribute="co2",
    sense="<=",
    constant=0.6 * co2_unbounded,
)
n_capped.optimize(include_objective_constant=False)

print(f"emissions:  {n_capped.statistics.effect('co2').sum():.0f} t "
      f"(cap {0.6 * co2_unbounded:.0f} t, before {co2_unbounded:.0f} t)")
print(f"shadow carbon price: {-n_capped.global_constraints.loc['co2_budget', 'mu']:.2f} EUR/t")

## Per-component coefficients

Carrier columns assign one coefficient to every member of a carrier. Quantities that belong to an *individual asset* — the land take of one specific wind site, the cooling water of one plant — go directly on the component, mirroring the two cost attributes: `marginal_effect_{name}` is the per-unit-of-operation analog of `marginal_cost`, and `capital_effect_{name}` the per-MW analog of `capital_cost`. Like all effects they are pure accounting until bounded or priced — here the site's land budget caps how much wind capacity can be built, and the land scarcity shows up as a shadow price.

In [ ]:
n_site = create_network()
n_site.generators.loc["wind farm", "p_nom_extendable"] = True
n_site.generators.loc["wind farm", "capital_cost"] = 60.0
n_site.generators.loc["wind farm", "capital_effect_land_use"] = 0.5  # ha/MW
n_site.generators.loc["gas turbine", "marginal_effect_water"] = 1.9  # m3/MWh
n_site.add("Effect", "land_use", unit="ha")
n_site.add("Effect", "water", unit="m3")
n_site.add(
    "GlobalConstraint",
    "site_area",
    type="effect_limit",
    carrier_attribute="land_use",
    sense="<=",
    constant=45.0,
)
n_site.optimize(include_objective_constant=False)

print(f"wind built: {n_site.generators.p_nom_opt['wind farm']:.1f} MW "
      f"(land cap 45 ha at 0.5 ha/MW)")
print(f"shadow price of site area: "
      f"{-n_site.global_constraints.loc['site_area', 'mu']:.0f} EUR/ha")

In [ ]:
n_site.statistics.effect("water", groupby=False)

## Pricing an effect

Instead of a hard cap, `price` routes an effect's value into the objective — a CO$_2$ price of 60 EUR/t as a declared input. This is exactly equivalent to manually folding `price × co2_emissions / efficiency` into every generator's marginal cost, which is what workflows like PyPSA-Eur do today by hand.

In [ ]:
n_priced = create_network()
n_priced.add("Effect", "co2", unit="t", carrier_attribute="co2_emissions", price=60.0)
n_priced.optimize(include_objective_constant=False)

co2_priced = n_priced.statistics.effect("co2").sum()
print(f"emissions at 60 EUR/t: {co2_priced:.0f} t (unpriced: {co2_unbounded:.0f} t)")
print(f"carbon cost share of objective: {60 * co2_priced / n_priced.objective:.1%}")

## Minimizing an effect

Any effect can *be* the objective. Here we minimize CO$_2$ subject to staying within 5% of the cost optimum — the cost cap is itself just an `effect_limit` on the built-in `cost` effect.

In [ ]:
cost_opt = n.objective

n_green = create_network()
n_green.add("Effect", "co2", unit="t", carrier_attribute="co2_emissions")
n_green.add(
    "GlobalConstraint",
    "cost_cap",
    type="effect_limit",
    carrier_attribute="cost",
    sense="<=",
    constant=1.05 * cost_opt,
)
n_green.optimize(objective_effect="co2", include_objective_constant=False)

print(f"minimal emissions within 5% cost slack: {n_green.objective:.0f} t "
      f"(cost-optimal dispatch emits {co2_unbounded:.0f} t)")

## The cost–emissions frontier

Sweeping the CO$_2$ cap between the two extremes traces the Pareto frontier between system cost and emissions — a few lines now that budgets are declarative.

In [ ]:
# the unconstrained emission minimum: minimize co2 with no cost consideration
m = create_network()
m.add("Effect", "co2", unit="t", carrier_attribute="co2_emissions")
m.optimize(objective_effect="co2", include_objective_constant=False)
co2_min = m.objective

caps = np.linspace(1.001 * co2_min, co2_unbounded, 9)
frontier = []
for cap in caps:
    m = create_network()
    m.add("Effect", "co2", unit="t", carrier_attribute="co2_emissions")
    m.add("GlobalConstraint", "co2_budget", type="effect_limit",
          carrier_attribute="co2", sense="<=", constant=cap)
    m.optimize(include_objective_constant=False)
    frontier.append((m.statistics.effect("co2").sum(), m.objective))
frontier = pd.DataFrame(frontier, columns=["co2", "cost"])

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(frontier.co2 / 1e3, frontier.cost / 1e3, marker="o", color="#2E7EB3")
ax.annotate("cost-optimal", frontier.iloc[-1] / 1e3,
            textcoords="offset points", xytext=(-8, 10), ha="right")
ax.annotate("CO$_2$-minimal", frontier.iloc[0] / 1e3,
            textcoords="offset points", xytext=(10, -4))
ax.set_xlabel("Emissions [kt]")
ax.set_ylabel("System cost [k EUR]")
plt.tight_layout()

## Summary

- Effects are declared like any component and **round-trip through IO**; coefficients ride on `n.carriers` columns exactly as `co2_emissions` always has, or per component via `marginal_effect_*` / `capital_effect_*` columns.
- **Reporting is always available** — per asset, per carrier, time-resolved — whether or not the effect appears in the model.
- An effect enters the model **only** when it is the objective, bounded (`effect_limit`), or priced (`price`); everything else is free bookkeeping. Effects add no decision variables, so an LP stays an LP.
- The classic CO$_2$ `primary_energy` constraint, carbon-price folding into marginal costs, and "minimize emissions under a cost cap" are all the same mechanism.